In [33]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [34]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

PTS | DuckDB connected to PostgreSQL


In [35]:
# ------------------------------------------------------------
# analytical-dataset construction context
# ------------------------------------------------------------

# the previous notebook selected the target and explanatory attributes
# this notebook turns that decision into a reproducible analytical dataset
# each row represents one eligible penguin observation
# the next steps define observation eligibility, then construct, validate and export the dataset

In [36]:
# ------------------------------------------------------------
# guiding construction questions — analytical-dataset
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# the following cells collect the evidence needed to answer them

# CQ1. which observations are eligible for the analytical dataset?
# CQ2. how does the eligibility rule affect the dataset size and target distribution?
# CQ3. does the constructed dataset preserve the intended observation-unit and data-grain?

In [37]:
# ------------------------------------------------------------
# construct the selected analytical source
# ------------------------------------------------------------

# retrieve the target and explanatory attributes selected in the previous analysis
# also recall the concepts of "observation-unit" and "data-grain"
#
# observation-unit:
# - one observation of one penguin
# - the same penguin may therefore contribute more than one observation
#
# data-grain:
# - defines the level of detail at which each row represents the observation-unit
# - here, each row must represent one specific penguin observed at one specific date_time
# - therefore, the intended data-grain is uniquely identified by (id_penguin, date_time)
#
# id_penguin and date_time were not selected as explanatory attributes
# they are temporarily retained to validate that the constructed data
# preserves the intended observation-unit and data-grain

db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_selected AS
SELECT
    o.id_penguin,
    o.date_time,

    -- TODO: include the candidate explanatory attributes
    --       selected in the previous analysis
    o.name_island as island,
    o.flipper_length_mm,
    o.bill_length_mm,
    o.bill_depth_mm,
    o.body_mass_g,
    EXTRACT( YEAR FROM o.date_time )::VARCHAR AS year,
    p.sex as sex,

    -- TODO: include the target attribute
    p.name_species as species

FROM pg.public.observation AS o JOIN pg.public.penguin AS p
     ON o.id_penguin = p.id;
""")

In [38]:
# ------------------------------------------------------------
# inspect missing values in the selected attributes
# ------------------------------------------------------------

# quantify missing values before defining the observation-eligibility rule
#
# recall:
# - COUNT(*) counts all rows
# - COUNT(attribute) counts only rows where attribute is not NULL
# - therefore, COUNT(*) - COUNT(attribute) gives the number of NULL values

db.sql("""
SELECT
    COUNT(*) AS n_observations,

    -- TODO: for each selected explanatory attribute and the target,
    --       count the number of missing values
    n_observations - count(year) as Na_year,
    n_observations - count(island) as Na_island,
    n_observations - count(flipper_length_mm) as Na_flipper_length,
    n_observations - count(bill_depth_mm) as Na_bill_depth,
    n_observations - count(bill_length_mm) as Na_bill_length,
    n_observations - count(body_mass_g) as Na_mass,
    n_observations - count(sex) as Na_sex,
    n_observations - count(species) as Na_species,

FROM penguins_selected;
""").show()

# expected result:
# - 344 observations
# - year, island and species have no missing values
# - sex has 11 missing values
# - each morphological measurement has 2 missing values

┌────────────────┬─────────┬───────────┬───────────────────┬───────────────┬────────────────┬─────────┬────────┬────────────┐
│ n_observations │ Na_year │ Na_island │ Na_flipper_length │ Na_bill_depth │ Na_bill_length │ Na_mass │ Na_sex │ Na_species │
│     int64      │  int64  │   int64   │       int64       │     int64     │     int64      │  int64  │ int64  │   int64    │
├────────────────┼─────────┼───────────┼───────────────────┼───────────────┼────────────────┼─────────┼────────┼────────────┤
│            344 │       0 │         0 │                 2 │             2 │              2 │       2 │     11 │          0 │
└────────────────┴─────────┴───────────┴───────────────────┴───────────────┴────────────────┴─────────┴────────┴────────────┘



In [ ]:
# ------------------------------------------------------------
# TODO
# conclude on CQ1 — observation eligibility
# ------------------------------------------------------------

# task:
# - use the missing-value counts obtained above
# - define which observations are eligible for the analytical dataset
# - consider the target and all selected explanatory attributes
# - for this analytical path, use a complete-case eligibility rule
# - id_penguin and date_time are retained only for validation and
#   do not determine observation eligibility

# conclusion for CQ1
# - an observation is eligible if:
#   none of the explanatory attributes have a null value

In [42]:
# ------------------------------------------------------------
# construct the analytical dataset
# ------------------------------------------------------------

# apply the observation-eligibility rule defined above
#
# id_penguin and date_time are not explanatory attributes
# and are therefore excluded from the analytical dataset
# they remain available in penguins_selected for the later validation
# of the intended observation-unit and data-grain

db.sql("""
CREATE OR REPLACE TEMP VIEW analytical_dataset AS
SELECT
    island,
    year,
    sex,
    bill_length_mm,
    bill_depth_mm,
    flipper_length_mm,
    body_mass_g,
    species
FROM penguins_selected
WHERE
    -- TODO: implement the complete-case eligibility rule
    --       for all selected explanatory attributes and the target
    island is not null and
    year is not null and
    sex is not null and
    flipper_length_mm is not null and
    bill_length_mm is not null and
    bill_depth_mm is not null and
    body_mass_g is not null;
""")

In [ ]:
# ------------------------------------------------------------
# validate the analytical dataset
# ------------------------------------------------------------

# check the number of eligible observations
db.sql("""
SELECT
    COUNT(*) AS n_observations
FROM analytical_dataset;
""").show()

# inspect the target distribution among eligible observations
db.sql("""
SELECT
    species,
    COUNT(*) AS n_observations
FROM analytical_dataset
GROUP BY species
ORDER BY n_observations DESC;
""").show()

┌────────────────┐
│ n_observations │
│     int64      │
├────────────────┤
│            333 │
└────────────────┘

┌───────────┬────────────────┐
│  species  │ n_observations │
│  varchar  │     int64      │
├───────────┼────────────────┤
│ Adelie    │            146 │
│ Gentoo    │            119 │
│ Chinstrap │             68 │
└───────────┴────────────────┘

┌───────────┬────────────────┐
│  species  │ n_observations │
│  varchar  │     int64      │
├───────────┼────────────────┤
│ Adelie    │            152 │
│ Gentoo    │            124 │
│ Chinstrap │             68 │
└───────────┴────────────────┘



In [46]:
# ------------------------------------------------------------
# validate observation-unit and data-grain
# ------------------------------------------------------------

# verify that each eligible row represents one observation-unit
# and that the intended data-grain is preserved:
# - one specific penguin observed at one specific date_time
#
# therefore, (id_penguin, date_time) must uniquely identify
# every eligible observation

db.sql("""
SELECT
    COUNT(*) AS n_observations,

    -- TODO: count the distinct observation-units identified by
    --       (id_penguin, date_time)
    count(distinct (id_penguin, date_time)) AS n_observation_units,

    -- TODO: compute how many observations violate the intended data-grain
    --       by repeating the same (id_penguin, date_time)
    n_observations - n_observation_units AS n_duplicate_observations

FROM penguins_selected
WHERE

    -- TODO: apply the same observation-eligibility rule
    --       used to construct analytical_dataset
    island is not null and
    year is not null and
    sex is not null and
    flipper_length_mm is not null and
    bill_length_mm is not null and
    bill_depth_mm is not null and
    body_mass_g is not null;
""").show()

# expected result:
# n_observations = 333
# n_observation_units = 333
# n_duplicate_observations = 0

┌────────────────┬─────────────────────┬──────────────────────────┐
│ n_observations │ n_observation_units │ n_duplicate_observations │
│     int64      │        int64        │          int64           │
├────────────────┼─────────────────────┼──────────────────────────┤
│            333 │                 333 │                        0 │
└────────────────┴─────────────────────┴──────────────────────────┘



In [60]:
# ------------------------------------------------------------
# compare target distribution before and after observation eligibility
# ------------------------------------------------------------

# use two Common Table Expressions (CTEs) to summarize the target
# distribution before and after applying the eligibility rule
#
# a CTE, defined with WITH, creates a named intermediate result
# that can be used within the same SQL statement

db.sql("""
WITH

before_eligibility AS (
SELECT
    species,
    COUNT(*) AS n_before,

    -- TODO: compute the percentage of each species before eligibility
    --       reuse the window-function pattern used previously
    100.0 * n_before / sum(n_before) over () AS pct_before

FROM penguins_selected
GROUP BY species
),

after_eligibility AS (
SELECT
    species,
    COUNT(*) AS n_after,

    -- TODO: compute the percentage of each species after eligibility
    100.0 * n_after / sum(n_after) over ()  AS pct_after

FROM analytical_dataset
GROUP BY species
)

SELECT
    b.species,
    b.n_before,
    a.n_after,
    ROUND(b.pct_before, 2) AS pct_before,
    ROUND(a.pct_after, 2) AS pct_after,

    -- TODO: compute the change in percentage points
    pct_before - pct_after AS percentage_point_difference

FROM before_eligibility AS b JOIN after_eligibility AS a
     ON b.species = a.species
ORDER BY b.species;
""").show()

# expected result:
# 3 rows, one for each species
# the largest absolute change in class percentage is less than 1 percentage point

┌───────────┬──────────┬─────────┬────────────┬───────────┬─────────────────────────────┐
│  species  │ n_before │ n_after │ pct_before │ pct_after │ percentage_point_difference │
│  varchar  │  int64   │  int64  │   double   │  double   │           double            │
├───────────┼──────────┼─────────┼────────────┼───────────┼─────────────────────────────┤
│ Adelie    │      152 │     146 │      44.19 │     43.84 │         0.34220266778406483 │
│ Chinstrap │       68 │      68 │      19.77 │     20.42 │         -0.6529785599553044 │
│ Gentoo    │      124 │     119 │      36.05 │     35.74 │           0.310775892171236 │
└───────────┴──────────┴─────────┴────────────┴───────────┴─────────────────────────────┘



In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for CQ2 and CQ3
# ------------------------------------------------------------

# task:
# - use the validation results obtained above
# - answer the evidence questions and conclude on CQ2 and CQ3

# E1. the individual missing-value counts sum to more than the number of excluded observations
# - what does this indicate about missing values across attributes?
# A1: That some observations have multiple missing-values (ex:. sex and flipper_length_mm), therefore being more than one to the sum but only one to the excluded observations.

# E2. by how many observations is the dataset reduced,
#     and what is the largest change in class percentage?
# A2: by 11.
#     the largest change is for Chinstrap, as the amount of rows is not reduces, while the others are.
#     This causes the difference to be negative (because the percentage of after reduction rows is higher than before reduction)
#     While the Chinstrap difference is negative, the reduction for the other two classes cannot be enough to overcome the difference
#     (if two go down by one and the other stays the same, the difference in percentages of the two is the inverse of the one that remains the same).

# conclusion for CQ2
# A: It reduces the dataset size (it can only make sure a row is not eligible, so it cannot add new eligible rows), 
#    and the distribution changes along with the reduction of the classes (if one class is reduced, the percentages of the others go up in distribution and vice-versa)

# E3. what does n_duplicate_observations indicate about
#     the intended observation-unit and data-grain?
# A3: That there are no repeated observation_units in the new dataset, therefore there are no data-grain violations in the dataset.

# E4. why are id_penguin and date_time useful for validation
#     but excluded from the analytical dataset?
# A4: id_penguin and date_time are identifiers, so each pair is specific to one observation of a penguin of id_penguin at time date_time.
#     This does not add information to the analysis of the dataset.

# conclusion for CQ3
# A: Yes, there are no duplicates.


#
# reminder (possibly useful for E4):
# observation-unit = what one record represents conceptually
# data-grain = the level of detail at which that observation-unit is represented
#
# in this MoP:
# observation-unit = one observation of one penguin
# data-grain = one specific penguin observed at one specific date_time
#              identified by (id_penguin, date_time)

In [ ]:
# ------------------------------------------------------------
# materialize the analytical dataset
# ------------------------------------------------------------

from pathlib import Path

output_file = Path("../03_output/penguins_analytical_dataset.csv").resolve()
output_file.parent.mkdir(parents=True, exist_ok=True)

db.sql(f"""
COPY analytical_dataset
TO '{ output_file.as_posix() }'
(FORMAT CSV, HEADER);
""")

print(output_file)

# Second visual checkpoint with Orange Data Mining (DM)

Before starting this checkpoint, make sure that the artefacts required from MoP_02
have been copied into the current MoP workspace.

In particular, you need:

- `01_input/_mLDm_00_penguins_data_collected.csv`
- `02_code/penguins_visual_check.ows`

The Orange workflow was created in MoP_02 and will now be extended rather than
recreated from scratch.

## 1. Reopen the workflow created in MoP_02

Open:

`02_code/penguins_visual_check.ows`

Keep the existing `File` widget connected to:

`01_input/_mLDm_00_penguins_data_collected.csv`

This branch corresponds to the initial collected source explored in notebook 02.

## 2. Add the analytical dataset

Add a second `File` widget and load, from the current MoP:

`03_output/penguins_analytical_dataset.csv`

Connect this new `File` widget to a new `Select Columns` widget.

For the analytical dataset, verify that:

- `species` is the **Target**
- the remaining attributes are **Features**
- attributes excluded during the previous analysis are no longer present

## 3. Compare both data representations

For each branch, connect `Select Columns` to:

- `Data Table`
- `Distributions`
- `Scatter Plot`

Use the two branches to compare:

- the attributes available in each dataset
- the roles assigned to those attributes
- the number and nature of the observations
- the visual distributions and relationships between selected attributes

Focus particularly on how the analytical dataset differs from the
initial collected source as a consequence of the decisions made in
notebooks 03 and 04.

Do not train or compare predictive models yet.

The purpose of this checkpoint is to compare the **collected source**
with the **materialized analytical dataset** before moving to the modelling stage.

Save the extended workflow again as:

`02_code/penguins_visual_check.ows`